Primeira tarefa do Job:
- Faz o download do arquivo no formato "zip" da Receite Federal, descompacta os arquivos "csv" e grava no volume da camada Bronze.

In [0]:
import requests
import zipfile
import os
import datetime as dt
import shutil

ano_mes_download = '2026-09' #dt.datetime.now().strftime("%Y-%m")
url = f"https://arquivos.receitafederal.gov.br/public.php/dav/files/YggdBLfdninEJX9/{ano_mes_download}/?accept=zip"

ano_mes_pasta = '2026_09' #dt.datetime.now().strftime("%Y_%m")
dbutils.jobs.taskValues.set(key="ano_mes_pasta", value=ano_mes_pasta)
volume_path = f"/Volumes/databricks_cnpj_data_lakehouse/bronze/{ano_mes_pasta}"
file_path = f"{volume_path}/raw_data_receita_federal.zip"

spark.sql("CREATE CATALOG IF NOT EXISTS databricks_cnpj_data_lakehouse")
spark.sql("CREATE SCHEMA IF NOT EXISTS databricks_cnpj_data_lakehouse.bronze")
spark.sql(f"CREATE VOLUME IF NOT EXISTS databricks_cnpj_data_lakehouse.bronze.`{ano_mes_pasta}`")

def baixar_arquivo(url, destino, max_tentativas=3):
    for tentativa in range(1, max_tentativas + 1):        
        try:
            with requests.get(url, stream=True, timeout=(10, 300)) as response:
                response.raise_for_status()
                bytes_baixados = 0
                proximo_log_mb = 500
                with open(destino, "wb") as f:
                    for chunk in response.iter_content(chunk_size=100 * 1024 * 1024):
                        if chunk:
                            f.write(chunk)
                            bytes_baixados += len(chunk)
                            mb_baixados = bytes_baixados / (1024 * 1024)
                            if mb_baixados >= proximo_log_mb:
                                print(f"Baixando... {mb_baixados:.0f} MB recebidos")
                                proximo_log_mb += 500
            
            with zipfile.ZipFile(destino, 'r') as z:
                corrompido = z.testzip()
            if corrompido is not None:
                raise zipfile.BadZipFile(f"Membro corrompido no zip baixado: {corrompido}")
            
            print(f"Download concluído e validado: {destino}")
            return True

        except (requests.exceptions.RequestException, zipfile.BadZipFile) as e:
            print(f"Tentativa {tentativa}/{max_tentativas} falhou: {e}")
            if os.path.exists(destino):
                os.remove(destino)
            if tentativa == max_tentativas:
                raise

def arquivo_valido(path):    
    if not os.path.exists(path):
        return False
    try:
        with zipfile.ZipFile(path, 'r') as z:
            return z.testzip() is None
    except zipfile.BadZipFile:
        return False

def unzip_files(file_path=None, validar_crc=True):
    if file_path is None:
        file_path = f"{volume_path}/raw_data_receita_federal.zip"

    print(f"Extraindo arquivos... {file_path}")

    if not os.path.exists(file_path):
        print(f"Arquivo não encontrado: {file_path}")
        return

    with zipfile.ZipFile(file_path, 'r') as z_master:
        if validar_crc:
            corrompido = z_master.testzip()
            if corrompido is not None:
                raise zipfile.BadZipFile(f"Membro corrompido no zip mestre: {corrompido}")

        nome_zip_atual = os.path.splitext(os.path.basename(file_path))[0]

        for info in z_master.infolist():
            if info.is_dir():
                continue

            nome_arquivo_interno = info.filename
            nome_limpo_interno = os.path.basename(nome_arquivo_interno)

            if nome_limpo_interno.startswith('.') or not nome_limpo_interno:
                continue

            nome_lower_interno = nome_limpo_interno.lower()

            if nome_lower_interno.endswith('.zip'):
                destino = f"{volume_path}/{nome_limpo_interno}"
            else:
                destino = f"{volume_path}/{nome_zip_atual}.csv"
            
            if not nome_lower_interno.endswith('.zip') and os.path.exists(destino):
                print(f"Já existe, pulando: {os.path.basename(destino)}")
                continue

            os.makedirs(os.path.dirname(destino), exist_ok=True)

            tmp_destino = destino + ".tmp"
            t0 = dt.datetime.now()
            try:
                with z_master.open(nome_arquivo_interno) as fin, open(tmp_destino, "wb") as fout:
                    shutil.copyfileobj(fin, fout, length=16 * 1024 * 1024)
                os.replace(tmp_destino, destino)
            except Exception:
                if os.path.exists(tmp_destino):
                    os.remove(tmp_destino)
                raise

            dt_s = (dt.datetime.now() - t0).total_seconds()
            tam_mb = info.file_size / (1024 * 1024)
            print(f"Extraído e salvo como: {os.path.basename(destino)} ({tam_mb:.1f} MB em {dt_s:.1f}s)")

            if nome_lower_interno.endswith('.zip'):
                unzip_files(destino, validar_crc=False)

    if not file_path.endswith('raw_data_receita_federal.zip'):
        print(f"Removendo arquivo zip processado: {file_path}")
        os.remove(file_path)

if arquivo_valido(file_path):
    print(f"Arquivo já baixado e válido: {file_path}")
else:
    if os.path.exists(file_path):
        print(f"Arquivo corrompido ou incompleto, removendo: {file_path}")
        dbutils.fs.rm(file_path)

    try:
        baixar_arquivo(url, file_path)        
    except Exception as e:
        print(f"Falha definitiva no download: {e}")
        print(f"Verifique no portal da Receita Federal se o arquivo do mês {ano_mes_download} já está disponível: https://arquivos.receitafederal.gov.br/index.php/s/YggdBLfdninEJX9")
        raise

try:
    unzip_files()
except Exception as e:
    print(f"Error: {e}")
    raise